# Technical tutor

The Week 1 question answerer, with a Gradio page, streaming, and a tutor system prompt. Answers come from the Azure OpenAI deployment in `.env`.

Bonus tools: token count and a small glossary in SQLite. Bonus audio: speak a question when `WHISPER_MODEL` is set, and hear the answer when `TTS_MODEL` is set.

## Clients

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

azure = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
azure_model = os.getenv("GPT_MODEL", "gpt-4o")
deployments = []
for name in (azure_model, "gpt-4o-mini", "gpt-4.1"):
    if name not in deployments:
        deployments.append(name)

import json
import sqlite3

import gradio as gr
import tiktoken

DB = repo_root / "W2-Build Multi Model Chatbots" / "glossary.db"

glossary = {
    "token": "A chunk of text the model reads or writes. Roughly four characters in English.",
    "embedding": "A list of numbers that represents the meaning of a piece of text.",
    "rag": "Retrieval-augmented generation: look up relevant notes, then answer from those notes.",
    "tool call": "A request from the model for our code to run a function, followed by the function result.",
    "prompt caching": "Reusing a repeated prompt prefix so later calls do not pay full price for that prefix.",
}

with sqlite3.connect(DB) as conn:
    conn.execute(
        "CREATE TABLE IF NOT EXISTS glossary (term TEXT PRIMARY KEY, definition TEXT)"
    )
    for term, definition in glossary.items():
        conn.execute(
            "INSERT INTO glossary (term, definition) VALUES (?, ?) "
            "ON CONFLICT(term) DO UPDATE SET definition = excluded.definition",
            (term, definition),
        )


## System prompt and tools

In [ ]:
system_prompt = """
You are a helpful technical tutor who answers questions about Python,
software engineering, data science, and LLMs.
Respond in markdown without wrapping the whole answer in a code block.
Be clear, structured, and practical.
Use count_tokens when the user asks how long a piece of text is.
Use lookup_term when the user asks what a glossary term means.
If a tool has no entry, say so.
"""


def count_tokens(text):
    encoding = tiktoken.encoding_for_model("gpt-4o")
    return f"{len(encoding.encode(text))} tokens, using the gpt-4o encoding."


def lookup_term(term):
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT definition FROM glossary WHERE term = ?",
            (term.lower().strip(),),
        ).fetchone()
    if not row:
        return f"No glossary entry for {term}."
    return row[0]


tools = [
    {
        "type": "function",
        "function": {
            "name": "count_tokens",
            "description": "Count tokens in a piece of text with the gpt-4o encoding.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "The text to count"},
                },
                "required": ["text"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_term",
            "description": "Look up a short definition in the local glossary.",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "Term such as token, embedding, or RAG",
                    },
                },
                "required": ["term"],
                "additionalProperties": False,
            },
        },
    },
]


def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        name = tool_call.function.name
        if name == "count_tokens":
            content = count_tokens(arguments.get("text", ""))
        elif name == "lookup_term":
            content = lookup_term(arguments.get("term", ""))
        else:
            content = f"Unknown tool {name}"
        responses.append(
            {
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            }
        )
    return responses


## Answer, then speak

Tool rounds complete before the answer is shown. If a model rejects tools, the except path streams a normal answer. Speech and transcription stay quiet until their Azure deployment names are in `.env`.

In [ ]:
def transcribe(audio_path):
    whisper_model = os.getenv("WHISPER_MODEL")
    if not whisper_model or not audio_path:
        return ""
    try:
        with open(audio_path, "rb") as audio_file:
            transcript = azure.audio.transcriptions.create(
                model=whisper_model,
                file=audio_file,
                timeout=120.0,
            )
        return transcript.text or ""
    except Exception as exc:
        print(f"Transcription failed: {exc}")
        return ""


def talker(message):
    tts_model = os.getenv("TTS_MODEL")
    if not tts_model or not message:
        return None
    try:
        speech = azure.audio.speech.create(
            model=tts_model,
            voice="alloy",
            input=message,
            timeout=120.0,
        )
        path = repo_root / "W2-Build Multi Model Chatbots" / "reply.mp3"
        path.write_bytes(speech.content)
        return str(path)
    except Exception as exc:
        print(f"Speech generation failed: {exc}")
        return None


def stream_answer(messages, deployment):
    try:
        response = azure.chat.completions.create(
            model=deployment,
            messages=messages,
            tools=tools,
            timeout=120.0,
        )
        while response.choices[0].finish_reason == "tool_calls":
            tool_message = response.choices[0].message
            messages.append(tool_message.model_dump(exclude_none=True))
            messages.extend(handle_tool_calls(tool_message))
            response = azure.chat.completions.create(
                model=deployment,
                messages=messages,
                tools=tools,
                timeout=120.0,
            )
        text = response.choices[0].message.content or ""
    except Exception as exc:
        print(f"Tool call failed, answering without tools: {exc}")
        stream = azure.chat.completions.create(
            model=deployment,
            messages=messages,
            stream=True,
            timeout=120.0,
        )
        text = ""
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                text += chunk.choices[0].delta.content
                yield text
        return

    partial = ""
    for word in text.split(" "):
        partial = word if not partial else f"{partial} {word}"
        yield partial


def respond(message, audio, history, deployment):
    history = list(history or [])
    spoken = transcribe(audio)
    if spoken:
        message = spoken
    message = (message or "").strip()
    if not message:
        yield history, None, ""
        return

    history = history + [{"role": "user", "content": message}]
    yield history, None, ""

    messages = [{"role": "system", "content": system_prompt}] + history
    shown = history
    partial = ""
    for partial in stream_answer(messages, deployment):
        shown = history + [{"role": "assistant", "content": partial}]
        yield shown, None, ""
    yield shown, talker(partial), ""


with gr.Blocks(title="Technical tutor") as demo:
    gr.Markdown("## Technical tutor")
    deployment = gr.Dropdown(deployments, value=azure_model, label="Model")
    chatbot = gr.Chatbot(height=480)
    audio_out = gr.Audio(label="Spoken answer", autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Question", lines=3, scale=4)
        audio_in = gr.Audio(label="Or speak", sources=["microphone"], type="filepath")
    message.submit(
        respond,
        inputs=[message, audio_in, chatbot, deployment],
        outputs=[chatbot, audio_out, message],
    )

demo.launch()
